# EnvBaseJ

> Base environment for JAX-backed CRLD environments.

In [ ]:
#| default_exp Environments/BaseJ

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
from fastcore.utils import *
import numpy as np
import jax
from jax import core
import jax.numpy as jnp

In [ ]:
#| export
class ebaseJ(object):
    """Base environment for JAX-backed CRLD environments."""

    def __init__(self):
        self.T = jnp.array(self.TransitionTensor())
        self.F = jnp.array(self.FinalStates())
        self.R = jnp.array(self.RewardTensor())
        self.O = jnp.array(self.ObservationTensor())

        self.Aset = self.actions()
        self.Sset = self.states()
        self.Oset = self.observations()

        # CHECKS
        R, T, O = self.R, self.T, self.O

        N = R.shape[0]
        assert O.shape[0] == N, "Inconsistent number of agents"
        assert len(T.shape[1:-1]) == N, "Inconsistent number of agents"
        assert len(R.shape[2:-1]) == N, "Inconsistent number of agents"

        M = T.shape[1]
        assert all(x == M for x in T.shape[1:-1]), 'Inconsistent number of actions'
        assert all(x == M for x in R.shape[2:-1]), 'Inconsistent number of actions'

        Z = T.shape[0]
        assert T.shape[-1] == Z, 'Inconsistent number of states'
        assert R.shape[-1] == Z, 'Inconsistent number of states'
        assert R.shape[1] == Z, 'Inconsistent number of states'
        assert O.shape[1] == Z, 'Inconsistent number of states'
        assert len(self.F) == Z, 'Inconsistent number of states'
        assert len(self.Sset) == Z, 'Inconsistent number of states'

        Q = O.shape[-1]
        assert np.allclose(list(map(len, self.Oset)),
                           np.array(Q).repeat(N)), 'Inconsistent number of observations'

        if not isinstance(T, core.Tracer):
            if not np.allclose(np.array(T).sum(-1), 1.0):
                raise ValueError("Transition model wrong")
        if not isinstance(O, core.Tracer):
            if not np.allclose(np.array(O).sum(-1), 1.0):
                raise ValueError("Observation model wrong")

In [ ]:
#| export
@patch
def TransitionTensor(self: ebaseJ):
    raise NotImplementedError

@patch
def RewardTensor(self: ebaseJ):
    raise NotImplementedError

In [ ]:
#| export
@patch
def ObservationTensor(self: ebaseJ):
    """Default observation tensor: perfect observation"""
    self.defaultObsTensUsed = True
    self.Q = self.Z
    Oiso = np.zeros((self.N, self.Z, self.Q))
    for i in range(self.N):
        Oiso[i, :, :] = np.eye(self.Q)
    return Oiso

In [ ]:
#| export
@patch
def FinalStates(self: ebaseJ):
    """Default final states: no final states."""
    return np.zeros(self.Z, dtype=int)

In [ ]:
#| export
@patch
def actions(self: ebaseJ):
    """Default action set representations."""
    return [[str(a) for a in range(self.M)] for _ in range(self.N)]

@patch
def states(self: ebaseJ):
    """Default state set representation."""
    return [str(s) for s in range(self.Z)]

@patch
def observations(self: ebaseJ):
    """Default observation set representations."""
    if hasattr(self, 'defaultObsTensUsed'):
        return [[str(o) for o in self.states()] for _ in range(self.N)]
    else:
        return [[str(o) for o in range(self.Q)] for _ in range(self.N)]

In [ ]:
#| export
@patch
def id(self: ebaseJ):
    """
    Returns id string of environment
    """
    # Default
    id = f"{self.__class__.__name__}"
    return id

@patch
def __str__(self: ebaseJ): return self.id()

@patch
def __repr__(self: ebaseJ): return self.id()

In [ ]:
#| export
@patch
def step(self: ebaseJ,
         jA: Iterable,  # joint actions
         key: jnp.ndarray = None  # JAX PRNG key
         ) -> tuple:  # (observations_i, rewards_i, done, info)
    """
    Iterate the environment one step forward.
    """
    tps = np.array(self.T[tuple([self.state] + list(jA))], dtype=float)
    tps = tps / tps.sum()

    if key is not None:
        key, subkey = jax.random.split(key)
        next_state = int(jax.random.choice(subkey, jnp.arange(len(tps)),
                                           p=jnp.array(tps)))
    else:
        next_state = np.random.choice(range(len(tps)), p=tps)

    rewards = self.R[tuple([slice(self.N), self.state] + list(jA) + [next_state])]

    self.state = next_state
    obs = self.observation(key=key)

    done = self.state in np.where(np.array(self.F) == 1)[0]
    info = {'state': self.state}

    return obs, np.array(rewards, dtype=float), done, info

In [ ]:
#| export
@patch
def observation(self: ebaseJ,
               key: jnp.ndarray = None  # JAX PRNG key
               ) -> np.ndarray:  # observations_i
    """
    Possibly random observation for each agent from the current state.
    """
    OBS = np.zeros(self.N, dtype=int)
    for i in range(self.N):
        ops = np.array(self.O[i, self.state], dtype=float)
        ops = ops / ops.sum()
        if key is not None:
            key, subkey = jax.random.split(key)
            obs = int(jax.random.choice(subkey, jnp.arange(len(ops)),
                                        p=jnp.array(ops)))
        else:
            obs = np.random.choice(range(len(ops)), p=ops)
        OBS[i] = obs
    return OBS

## Tests

In [ ]:
# Basic check: ebaseJ cannot be instantiated directly (abstract)
try:
    e = ebaseJ()
    print('ERROR: should have raised NotImplementedError')
except (NotImplementedError, AttributeError):
    print('ebaseJ correctly abstract: OK')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()